In [18]:
!pip install tabulate


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


# Dependencies

In [19]:
!pip install psycopg2-binary sqlalchemy pandas
!pip install scikit-learn matplotlib seaborn


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [20]:
!pip install colorama
!pip install requests
!pip install tqdm
!pip install nbformat



[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [21]:

!pip install jinja2


[notice] A new release of pip is available: 25.1.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


In [1]:
import pandas as pd
from sqlalchemy import create_engine
import tools.helpers as hh


In [2]:
user = "postgres"
password = "mimic"
host = "localhost"   
port = "5432"
database = "mimiciv"

In [3]:
engine = create_engine(f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{database}")


## RS diseases ICU

In [4]:
query = """

SELECT DISTINCT
  i.subject_id,
  i.hadm_id,
  i.stay_id,
  di.icd_version,
  UPPER(di.icd_code) AS icd_code
FROM mimiciv_icu.icustays AS i
JOIN mimiciv_hosp.diagnoses_icd AS di
  ON di.subject_id = i.subject_id
 AND di.hadm_id    = i.hadm_id
WHERE
  (di.icd_version = 10 AND LEFT(UPPER(di.icd_code),1) = 'J')
  OR
  (di.icd_version = 9
   AND LEFT(di.icd_code,1) ~ '^[0-9]$'
   AND CAST(LEFT(di.icd_code,3) AS INT) BETWEEN 460 AND 519)
ORDER BY i.subject_id, i.hadm_id, i.stay_id, di.icd_version, icd_code;

"""
rs_icu_df = pd.read_sql(query, engine)
hh.dx(rs_icu_df)

36.2k Unique Patient IDs (36184)
45.7k Unique Admission IDs (45670)
52.7k Unique ICU Stay IDs (52697)
110.2k Rows, shape: (110195, 5)



In [6]:
pd.read_sql("""SELECT * FROM mimiciv_hosp.diagnoses_icd LIMIT 4""", engine)

,subject_id,hadm_id,seq_num,icd_code,icd_version
0,10000032,22595853,1,5723,9
1,10000032,22595853,2,78959,9
2,10000032,22595853,3,5715,9
3,10000032,22595853,4,07070,9


## Respiratory infections overall

In [8]:
query = """

SELECT DISTINCT di.subject_id, 
di.hadm_id, 
di.icd_version, 
UPPER(di.icd_code) AS icd_code
FROM mimiciv_hosp.diagnoses_icd di
WHERE
  (di.icd_version=10 AND UPPER(di.icd_code) ~ '^(J0[0-6]|J09|J1[0-8]|J2[0-2]|J8[56])')
  OR
  (di.icd_version=9  AND di.icd_code ~ '^(46[0-6]|48[0-8])')
"""
rs_inf_df = pd.read_sql_query(query, con=engine)



rs_inf_df = pd.read_sql(query, engine)
hh.dx(rs_inf_df)

27.5k Unique Patient IDs (27480)
37.6k Unique Admission IDs (37627)
ICU Stay ID column not found.
40.2k Rows, shape: (40235, 4)



In [17]:
hh.dxx(rs_inf_df)

27.5k Unique Patient IDs (27480)
37.6k Unique Admission IDs (37627)
ICU Stay ID column not found.
40.2k Rows, shape: (40235, 4)



,subject_id,hadm_id,icd_version,icd_code
dtype,int64,int64,int64,object
NotNA | NA,40235 | 0,40235 | 0,40235 | 0,40235 | 0
nunique,27480,37627,2,164
0,10000690,23280645,9,486
1,10000826,20032235,9,486
2,10000904,28328117,9,463


In [16]:
rs_inf_df

,subject_id,hadm_id,icd_version,icd_code
0,10000690,23280645,9,486
1,10000826,20032235,9,486
2,10000904,28328117,9,463
3,10001176,23334588,9,4829
4,10001843,26133978,10,J189
...,...,...,...,...
40230,19999112,24284132,10,J029
40231,19999287,20175828,9,4829
40232,19999287,22997012,9,486
40233,19999625,25304202,9,486


In [18]:
hh.dxx(rs_inf_df)

27.5k Unique Patient IDs (27480)
37.6k Unique Admission IDs (37627)
ICU Stay ID column not found.
40.2k Rows, shape: (40235, 4)



,subject_id,hadm_id,icd_version,icd_code
dtype,int64,int64,int64,object
NotNA | NA,40235 | 0,40235 | 0,40235 | 0,40235 | 0
nunique,27480,37627,2,164
0,10000690,23280645,9,486
1,10000826,20032235,9,486
2,10000904,28328117,9,463


## Respiratory infections ICU

In [59]:
query = r"""
SELECT DISTINCT a.*
FROM mimiciv_hosp.patients p
JOIN mimiciv_hosp.admissions a
  ON p.subject_id = a.subject_id

"""
test_hosp = pd.read_sql_query(query, con=engine)
hh.dxx(test_hosp)

223.5k Unique Patient IDs (223452)
546.0k Unique Admission IDs (546028)
ICU Stay ID column not found.
546.0k Rows, shape: (546028, 16)



,subject_id,hadm_id,admittime,dischtime,deathtime,admission_type,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
dtype,int64,int64,datetime64[ns],datetime64[ns],datetime64[ns],object,object,object,object,object,object,object,object,datetime64[ns],datetime64[ns],int64
NotNA | NA,546028 | 0,546028 | 0,546028 | 0,546028 | 0,11790 | 534238,546028 | 0,546024 | 4,546027 | 1,396210 | 149818,536673 | 9355,545253 | 775,532409 | 13619,546028 | 0,379240 | 166788,379240 | 166788,546028 | 0
nunique,223452,546028,534919,528871,11789,9,2046,12,14,6,26,5,33,372693,372756,2
0,10000032,22595853,2180-05-06 22:23:00,2180-05-07 17:15:00,NaT,URGENT,P49AFC,TRANSFER FROM HOSPITAL,HOME,Medicaid,English,WIDOWED,WHITE,2180-05-06 19:17:00,2180-05-06 23:30:00,0
1,10000032,22841357,2180-06-26 18:27:00,2180-06-27 18:49:00,NaT,EW EMER.,P784FA,EMERGENCY ROOM,HOME,Medicaid,English,WIDOWED,WHITE,2180-06-26 15:54:00,2180-06-26 21:31:00,0
2,10000032,25742920,2180-08-05 23:44:00,2180-08-07 17:50:00,NaT,EW EMER.,P19UTS,EMERGENCY ROOM,HOSPICE,Medicaid,English,WIDOWED,WHITE,2180-08-05 20:58:00,2180-08-06 01:44:00,0


In [67]:
query = r"""
SELECT p.*,a.*
FROM mimiciv_hosp.patients p
JOIN mimiciv_hosp.admissions a
  ON p.subject_id = a.subject_id

"""
test_hosp_w = pd.read_sql_query(query, con=engine)
# hh.dxx(test_hosp)

In [68]:
test_hosp_w

,subject_id,gender,anchor_age,anchor_year,anchor_year_group,dod,subject_id,hadm_id,admittime,dischtime,...,admit_provider_id,admission_location,discharge_location,insurance,language,marital_status,race,edregtime,edouttime,hospital_expire_flag
0,10000886,M,18,2178,2008 - 2010,None,10000886,21927847,2178-05-08 08:07:00,2178-05-08 11:46:00,...,P24H0T,EMERGENCY ROOM,None,None,English,SINGLE,UNABLE TO OBTAIN,2178-05-08 03:40:00,2178-05-08 11:46:00,0
1,10000947,M,60,2121,2020 - 2022,2121-08-23,10000947,27880650,2121-05-09 10:02:00,2121-05-13 16:20:00,...,P690UI,TRANSFER FROM HOSPITAL,HOME,Private,English,MARRIED,BLACK/AFRICAN AMERICAN,2121-05-09 07:25:00,2121-05-09 15:01:00,0
2,10001319,F,28,2133,2008 - 2010,None,10001319,23005466,2135-07-20 03:45:00,2135-07-22 11:38:00,...,P94QZF,PHYSICIAN REFERRAL,HOME,Private,English,MARRIED,WHITE,NaT,NaT,0
3,10001319,F,28,2133,2008 - 2010,None,10001319,24591241,2138-11-09 20:00:00,2138-11-12 10:40:00,...,P94QZF,PHYSICIAN REFERRAL,HOME,Private,English,MARRIED,WHITE,NaT,NaT,0
4,10001319,F,28,2133,2008 - 2010,None,10001319,29230609,2134-04-15 07:59:00,2134-04-17 13:23:00,...,P94QZF,PHYSICIAN REFERRAL,HOME,Private,English,MARRIED,WHITE,NaT,NaT,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
546023,19997887,F,57,2112,2011 - 2013,None,19997887,25047276,2117-04-07 00:00:00,2117-04-09 16:10:00,...,P20GPE,PHYSICIAN REFERRAL,HOME HEALTH CARE,Private,English,MARRIED,WHITE,NaT,NaT,0
546024,19998350,M,52,2127,2011 - 2013,None,19998350,21130518,2131-04-26 18:43:00,2131-04-27 15:44:00,...,P740G6,EMERGENCY ROOM,None,Medicaid,English,MARRIED,BLACK/AFRICAN AMERICAN,2131-04-25 20:34:00,2131-04-26 20:17:00,0
546025,19998350,M,52,2127,2011 - 2013,None,19998350,27108332,2128-02-21 09:41:00,2128-02-22 17:04:00,...,P948LP,EMERGENCY ROOM,None,Medicare,English,SINGLE,BLACK/AFRICAN AMERICAN,2128-02-21 08:08:00,2128-02-21 11:54:00,0
546026,19999156,F,62,2123,2011 - 2013,None,19999156,22917423,2123-01-27 15:47:00,2123-01-27 21:39:00,...,P47DQU,EMERGENCY ROOM,None,Private,English,MARRIED,WHITE,2123-01-27 13:13:00,2123-01-27 21:39:00,0


In [62]:
test_hosp.admission_type.value_counts()

admission_type
EW EMER.                       177459
EU OBSERVATION                 119456
OBSERVATION ADMIT               84437
URGENT                          54929
SURGICAL SAME DAY ADMISSION     42898
DIRECT OBSERVATION              24551
DIRECT EMER.                    21973
ELECTIVE                        13130
AMBULATORY OBSERVATION           7195
Name: count, dtype: int64

In [63]:
test_hosp.admission_location.value_counts()


admission_location
EMERGENCY ROOM                            244179
PHYSICIAN REFERRAL                        163228
TRANSFER FROM HOSPITAL                     56227
WALK-IN/SELF REFERRAL                      42365
CLINIC REFERRAL                            12965
PROCEDURE SITE                              8518
TRANSFER FROM SKILLED NURSING FACILITY      6317
INTERNAL TRANSFER TO OR FROM PSYCH          5837
PACU                                        5734
INFORMATION NOT AVAILABLE                    402
AMBULATORY SURGERY TRANSFER                  255
Name: count, dtype: int64

In [65]:
test_hosp.describe()

,subject_id,hadm_id,admittime,dischtime,deathtime,edregtime,edouttime,hospital_expire_flag
count,5.460280e+05,5.460280e+05,546028,546028,11790,379240,379240,546028.000000
mean,1.501118e+07,2.500100e+07,2155-02-18 15:33:10.993172480,2155-02-23 09:50:05.473454080,2153-10-10 11:43:57.603053568,2155-07-14 16:55:15.539972096,2155-07-15 03:48:02.511442944,0.021612
min,1.000003e+07,2.000002e+07,2105-10-04 17:26:00,2105-10-12 11:11:00,2110-01-25 09:40:00,2106-02-06 15:47:00,2106-02-07 09:31:00,0.000000
25%,1.252380e+07,2.249662e+07,2135-02-14 13:13:30,2135-02-18 17:15:15.000000512,2133-10-02 13:43:15.000000512,2135-07-14 13:16:00,2135-07-14 21:32:15.000000512,0.000000
50%,1.501961e+07,2.500385e+07,2155-01-11 17:56:00,2155-01-16 13:38:30,2153-11-01 23:12:30,2155-06-11 20:23:00,2155-06-12 06:27:30,0.000000
75%,1.750403e+07,2.750282e+07,2175-03-15 15:55:30,2175-03-19 15:11:00,2173-10-06 11:06:44.999999488,2175-08-05 19:00:30,2175-08-06 10:40:15.000000512,0.000000
max,1.999999e+07,2.999994e+07,2214-12-15 19:11:00,2214-12-24 13:44:00,2214-10-12 12:51:00,2214-12-15 00:45:00,2214-12-15 22:50:00,1.000000
std,2.877694e+06,2.888710e+06,NaN,NaN,NaN,NaN,NaN,0.145415


In [46]:
query = r"""
SELECT *
FROM mimiciv_icu.icustays i

"""
test = pd.read_sql_query(query, con=engine)
hh.dxx(test)

65.4k Unique Patient IDs (65366)
85.2k Unique Admission IDs (85242)
94.5k Unique ICU Stay IDs (94458)
94.5k Rows, shape: (94458, 8)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64
NotNA | NA,94458 | 0,94458 | 0,94458 | 0,94458 | 0,94458 | 0,94458 | 0,94444 | 14,94444 | 14
nunique,65366,85242,94458,17,17,94449,94443,84933
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535


In [61]:
y= test[test['hadm_id'].isin(test_hosp['hadm_id'])]
hh.dxx(y)

65.4k Unique Patient IDs (65366)
85.2k Unique Admission IDs (85242)
94.5k Unique ICU Stay IDs (94458)
94.5k Rows, shape: (94458, 8)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64
NotNA | NA,94458 | 0,94458 | 0,94458 | 0,94458 | 0,94458 | 0,94458 | 0,94444 | 14,94444 | 14
nunique,65366,85242,94458,17,17,94449,94443,84933
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535


In [66]:
y

,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los
0,10000032,29079034,39553978,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2180-07-23 14:00:00,2180-07-23 23:50:47,0.410266
1,10000690,25860671,37081114,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2150-11-02 19:37:00,2150-11-06 17:03:17,3.893252
2,10000980,26913865,39765666,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2189-06-27 08:42:00,2189-06-27 20:38:27,0.497535
3,10001217,24597018,37067082,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-11-20 19:18:02,2157-11-21 22:08:00,1.118032
4,10001217,27703517,34592300,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2157-12-19 15:42:24,2157-12-20 14:27:41,0.948113
...,...,...,...,...,...,...,...,...
94453,19999442,26785317,32336619,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2148-11-19 14:23:43,2148-11-26 13:12:15,6.950370
94454,19999625,25304202,31070865,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2139-10-10 19:18:00,2139-10-11 18:21:28,0.960741
94455,19999828,25744818,36075953,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2149-01-08 18:12:00,2149-01-10 13:11:02,1.790995
94456,19999840,21033226,38978960,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2164-09-12 09:26:28,2164-09-17 16:35:15,5.297766


In [69]:
query = r"""
SELECT DISTINCT i.subject_id, i.hadm_id, i.stay_id,di.seq_num,
       di.icd_version, UPPER(di.icd_code) AS icd_code
FROM mimiciv_icu.icustays i
JOIN mimiciv_hosp.diagnoses_icd di
  ON di.subject_id = i.subject_id
 AND di.hadm_id    = i.hadm_id
WHERE
  ((di.icd_version=10 AND UPPER(di.icd_code) ~ '^(J0[0-6]|J09|J1[0-8]|J2[0-2]|J8[56])')
  OR
  (di.icd_version=9  AND di.icd_code ~ '^(46[0-6]|48[0-8])'))
  AND (seq_num=1 or seq_num=2) 
"""
df_resp_inf_icu_prim_sec = pd.read_sql_query(query, con=engine)
hh.dxx(df_resp_inf_icu_prim_sec)

5.2k Unique Patient IDs (5201)
5.7k Unique Admission IDs (5670)
6.5k Unique ICU Stay IDs (6454)
6.5k Rows, shape: (6490, 6)



,subject_id,hadm_id,stay_id,seq_num,icd_version,icd_code
dtype,int64,int64,int64,int64,int64,object
NotNA | NA,6490 | 0,6490 | 0,6490 | 0,6490 | 0,6490 | 0,6490 | 0
nunique,5201,5670,6454,2,2,116
0,10002155,20345487,32358465,2,9,486
1,10002155,23822395,33685454,2,9,486
2,10002155,28994087,31090461,1,9,486


In [10]:
query = r"""
SELECT DISTINCT i.subject_id, i.hadm_id, i.stay_id,di.seq_num,
       di.icd_version, UPPER(di.icd_code) AS icd_code
FROM mimiciv_icu.icustays i
JOIN mimiciv_hosp.diagnoses_icd di
  ON di.subject_id = i.subject_id
 AND di.hadm_id    = i.hadm_id
WHERE
  (di.icd_version=10 AND UPPER(di.icd_code) ~ '^(J0[0-6]|J09|J1[0-8]|J2[0-2]|J8[56])')
  OR
  (di.icd_version=9  AND di.icd_code ~ '^(46[0-6]|48[0-8])')

"""
df_resp_inf_icu = pd.read_sql_query(query, con=engine)
hh.dxx(df_resp_inf_icu)

11.9k Unique Patient IDs (11905)
13.6k Unique Admission IDs (13611)
16.4k Unique ICU Stay IDs (16438)
18.1k Rows, shape: (18109, 6)


13.6k Unique Admission IDs (13611)
16.4k Unique ICU Stay IDs (16438)
18.1k Rows, shape: (18109, 6)



,subject_id,hadm_id,stay_id,seq_num,icd_version,icd_code
dtype,int64,int64,int64,int64,int64,object
NotNA | NA,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0
nunique,11905,13611,16438,38,2,142
0,10001843,26133978,39698942,4,10,J189
1,10001884,26184834,37510196,19,10,J0190
2,10002155,20345487,32358465,2,9,486


In [71]:
df_resp_inf_icu

,subject_id,hadm_id,stay_id,seq_num,icd_version,icd_code
0,10001843,26133978,39698942,4,10,J189
1,10001884,26184834,37510196,19,10,J0190
2,10002155,20345487,32358465,2,9,486
3,10002155,23822395,33685454,2,9,486
4,10002155,28994087,31090461,1,9,486
...,...,...,...,...,...,...
18104,19998330,24492004,32641669,3,9,486
18105,19998878,29356037,30532790,2,9,486
18106,19999287,20175828,35165301,2,9,4829
18107,19999287,22997012,37692584,2,9,486


In [72]:
hh.dxx(df_resp_inf_icu)

11.9k Unique Patient IDs (11905)
13.6k Unique Admission IDs (13611)
16.4k Unique ICU Stay IDs (16438)
18.1k Rows, shape: (18109, 6)



,subject_id,hadm_id,stay_id,seq_num,icd_version,icd_code
dtype,int64,int64,int64,int64,int64,object
NotNA | NA,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0,18109 | 0
nunique,11905,13611,16438,38,2,142
0,10001843,26133978,39698942,4,10,J189
1,10001884,26184834,37510196,19,10,J0190
2,10002155,20345487,32358465,2,9,486


In [ ]:
# df_resp_inf_icu.to_parquet("df_resp_inf_icu_final.parq", engine="fastparquet")


In [26]:
df_resp_inf_icu= pd.read_parquet("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/parq/df_resp_inf_icu_final.parq", engine="fastparquet")

In [28]:
df_resp_inf_icu_ids = tuple(int(x) for x in df_resp_inf_icu.subject_id.unique())

In [29]:
df_resp_inf_icu_hadm_ids = tuple(int(x) for x in df_resp_inf_icu.hadm_id.unique())

In [30]:
df_resp_inf_icu_stay_ids = tuple(int(x) for x in df_resp_inf_icu.stay_id.unique())


### LAB

In [ ]:
# using hadm_id instead of subject_id to get lab events for that particular hospital admission
# then we can filter further based on stay_id and icu intime outtime to get lab events during icu stay

In [77]:

query= f"""SELECT 
    le.subject_id, le.hadm_id, le.labevent_id, le.specimen_id, le.itemid,
    dli.label , dli.fluid , dli.category,
    le.charttime, le.storetime, le.value, le.valuenum, le.valueuom,
    le.ref_range_lower, le.ref_range_upper, le.flag, le.priority, le.comments
FROM mimiciv_hosp.labevents le
LEFT JOIN mimiciv_hosp.d_labitems dli ON le.itemid = dli.itemid
WHERE le.hadm_id IS NOT NULL 
AND le.hadm_id in {df_resp_inf_icu_hadm_ids};"""


resp_lab_test_icu_df = pd.read_sql_query(query, con=engine)

In [38]:
query = f"""
SELECT
    icu.subject_id, icu.hadm_id, icu.stay_id, icu.first_careunit, icu.last_careunit,
    icu.intime, icu.outtime, icu.los,
    le.labevent_id, le.specimen_id, le.itemid,
    dli.label, dli.fluid, dli.category,
    le.charttime, le.storetime, le.value, le.valuenum, le.valueuom,
    le.ref_range_lower, le.ref_range_upper, le.flag, le.priority, le.comments
FROM mimiciv_icu.icustays icu
INNER JOIN mimiciv_hosp.labevents le
  ON icu.subject_id = le.subject_id
  AND icu.hadm_id   = le.hadm_id
  
LEFT JOIN mimiciv_hosp.d_labitems dli
  ON le.itemid = dli.itemid
WHERE icu.hadm_id IN {df_resp_inf_icu_hadm_ids}
  AND icu.hadm_id IS NOT NULL
  AND le.charttime IS NOT NULL
"""
lab_test_hosp_icu_df = pd.read_sql_query(query, con=engine)


In [ ]:
# hh.parq(lab_test_hosp_icu_df,'lab_test_hosp_icu_df_')

File saved at: lab_test_hosp_icu_df_23Jan26_2125.parquet


In [41]:
lab_test_hosp_icu_df= hh.load_data('./parq/lab_test_hosp_icu_df_23Jan26_2125.parquet')
hh.dxx(lab_test_hosp_icu_df)

11.9k Unique Patient IDs (11864)
13.5k Unique Admission IDs (13549)
16.4k Unique ICU Stay IDs (16370)
16.2M Rows, shape: (16160654, 24)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,labevent_id,specimen_id,itemid,label,fluid,category,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,int64,int64,object,object,object,datetime64[ns],datetime64[ns],object,float64,object,float64,float64,object,object,object
NotNA | NA,16160654 | 0,16160654 | 0,16160654 | 0,16160654 | 0,16160654 | 0,16160654 | 0,16159695 | 959,16159695 | 959,16160654 | 0,16160654 | 0,16160654 | 0,16160654 | 0,16160654 | 0,16160654 | 0,16160654 | 0,16102997 | 57657,15288675 | 871979,14732259 | 1428395,13899108 | 2261546,13207329 | 2953325,13207329 | 2953325,6803607 | 9357047,13562976 | 2597678,2393141 | 13767513
nunique,11864,13549,16370,14,14,16370,16367,16185,10886372,1158463,848,688,9,3,682239,1601059,15045,18070,57,118,170,2,3,5100
0,10015625,22068925,38098599,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2177-02-11 12:46:10,2177-02-17 17:52:53,6.212998,265253,73252082,50802,Base Excess,Blood,Blood Gas,2177-02-15 19:55:00,2177-02-15 20:20:00,-4,-4.000000,mEq/L,nan,nan,None,None,None
1,10015625,22068925,38098599,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2177-02-11 12:46:10,2177-02-17 17:52:53,6.212998,265573,2360138,50947,I,Blood,Chemistry,2177-02-23 06:16:00,2177-02-23 08:20:00,1,1.000000,None,nan,nan,None,ROUTINE,None
2,10015625,22068925,38098599,Surgical Intensive Care Unit (SICU),Surgical Intensive Care Unit (SICU),2177-02-11 12:46:10,2177-02-17 17:52:53,6.212998,264734,15757621,50868,Anion Gap,Blood,Chemistry,2177-02-06 20:43:00,2177-02-06 23:27:00,11,11.000000,mEq/L,10.000000,18.000000,None,STAT,None


In [14]:
query = f"""
SELECT
    icu.subject_id, icu.hadm_id, icu.stay_id, icu.first_careunit, icu.last_careunit,
    icu.intime, icu.outtime, icu.los,
    le.labevent_id, le.specimen_id, le.itemid,
    dli.label, dli.fluid, dli.category,
    le.charttime, le.storetime, le.value, le.valuenum, le.valueuom,
    le.ref_range_lower, le.ref_range_upper, le.flag, le.priority, le.comments
FROM mimiciv_icu.icustays icu
INNER JOIN mimiciv_hosp.labevents le
  ON icu.subject_id = le.subject_id
  AND icu.hadm_id   = le.hadm_id
  AND le.charttime BETWEEN icu.intime AND icu.outtime
LEFT JOIN mimiciv_hosp.d_labitems dli
  ON le.itemid = dli.itemid
WHERE icu.hadm_id IN {df_resp_inf_icu_hadm_ids}
  AND icu.hadm_id IS NOT NULL
  AND le.charttime IS NOT NULL
"""
lab_test_icu_df = pd.read_sql_query(query, con=engine)


In [ ]:
# lab_test_icu_df.to_parquet("lab_test_icu_df.parq", engine="fastparquet")

In [15]:
hh.dxx(lab_test_icu_df)

11.8k Unique Patient IDs (11782)

13.4k Unique Admission IDs (13443)
13.4k Unique Admission IDs (13443)
16.1k Unique ICU Stay IDs (16133)
6.9M Rows, shape: (6867008, 24)

16.1k Unique ICU Stay IDs (16133)
6.9M Rows, shape: (6867008, 24)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,labevent_id,specimen_id,itemid,label,fluid,category,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,int64,int64,object,object,object,datetime64[ns],datetime64[ns],object,float64,object,float64,float64,object,object,object
NotNA | NA,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6867008 | 0,6843544 | 23464,6544501 | 322507,6205278 | 661730,5833872 | 1033136,5459869 | 1407139,5459869 | 1407139,2852209 | 4014799,5192206 | 1674802,1079921 | 5787087
nunique,11782,13443,16133,14,14,16133,16133,15951,6867008,760903,828,676,9,3,481148,1106584,12708,15329,57,114,165,2,3,4112
0,10001843,26133978,39698942,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2134-12-05 18:50:03,2134-12-06 14:38:26,0.825266,17153,44776621,50802,Base Excess,Blood,Blood Gas,2134-12-05 19:30:00,2134-12-05 19:52:00,1,1.000000,mEq/L,nan,nan,None,None,None
1,10008924,23676183,30244392,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2139-07-09 23:42:19,2139-07-11 17:04:07,1.723472,143897,42163508,50861,Alanine Aminotransferase (ALT),Blood,Chemistry,2139-07-10 05:20:00,2139-07-10 06:36:00,68,68.000000,IU/L,0.000000,40.000000,abnormal,ROUTINE,None
2,10008924,23676183,30244392,Medical Intensive Care Unit (MICU),Medical Intensive Care Unit (MICU),2139-07-09 23:42:19,2139-07-11 17:04:07,1.723472,143898,42163508,50862,Albumin,Blood,Chemistry,2139-07-10 05:20:00,2139-07-10 06:36:00,3.8,3.800000,g/dL,3.500000,5.200000,None,ROUTINE,None


In [78]:
hh.dxx(resp_lab_test_icu_df)


11.9k Unique Patient IDs (11864)
13.5k Unique Admission IDs (13549)
ICU Stay ID column not found.
10.9M Rows, shape: (10886372, 18)



,subject_id,hadm_id,labevent_id,specimen_id,itemid,label,fluid,category,charttime,storetime,value,valuenum,valueuom,ref_range_lower,ref_range_upper,flag,priority,comments
dtype,int64,int64,int64,int64,int64,object,object,object,datetime64[ns],datetime64[ns],object,float64,object,float64,float64,object,object,object
NotNA | NA,10886372 | 0,10886372 | 0,10886372 | 0,10886372 | 0,10886372 | 0,10886372 | 0,10886372 | 0,10886372 | 0,10886372 | 0,10846170 | 40202,10287991 | 598381,9895023 | 991349,9352168 | 1534204,8869937 | 2016435,8869937 | 2016435,4497708 | 6388664,9083523 | 1802849,1632399 | 9253973
nunique,11864,13549,10886372,1158463,848,688,9,3,682239,1601059,15045,18070,57,118,170,2,3,5100
0,10001843,26133978,17043,80784905,50861,Alanine Aminotransferase (ALT),Blood,Chemistry,2134-12-05 09:08:00,2134-12-05 10:13:00,77,77.000000,IU/L,0.000000,40.000000,abnormal,STAT,None
1,10001843,26133978,17044,80784905,50862,Albumin,Blood,Chemistry,2134-12-05 09:08:00,2134-12-05 10:13:00,2.4,2.400000,g/dL,3.500000,5.200000,abnormal,STAT,None
2,10001843,26133978,17045,80784905,50863,Alkaline Phosphatase,Blood,Chemistry,2134-12-05 09:08:00,2134-12-05 10:13:00,569,569.000000,IU/L,40.000000,130.000000,abnormal,STAT,None


In [83]:
!pip install pyarrow
!pip install fastparquet

  Using cached fastparquet-2024.11.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (4.2 kB)
  Using cached cramjam-2.11.0-cp313-cp313-macosx_11_0_arm64.whl.metadata (5.6 kB)
  Using cached fsspec-2025.9.0-py3-none-any.whl.metadata (10 kB)
Using cached fastparquet-2024.11.0-cp313-cp313-macosx_11_0_arm64.whl (683 kB)
Using cached cramjam-2.11.0-cp313-cp313-macosx_11_0_arm64.whl (1.7 MB)
Using cached fsspec-2025.9.0-py3-none-any.whl (199 kB)
   25l━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [fastparquet]


In [116]:
resp_lab_test_icu_df.to_parquet("resp_lab_test_icu_inf_df.parq", engine="fastparquet")

### MICRO

In [31]:
query = f"""
-- Get respiratory infection ICU patients with any microbiology record during ICU stay (raw data)
SELECT 
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    icu.first_careunit,
    icu.last_careunit,
    icu.intime,
    icu.outtime,
    icu.los,
    me.microevent_id,
    me.chartdate,
    me.charttime,
    me.spec_type_desc,
    me.test_name,
    me.org_name,
    me.ab_name,
    me.interpretation,
    me.dilution_value,
    me.comments
FROM mimiciv_icu.icustays icu
INNER JOIN mimiciv_hosp.microbiologyevents me 
    ON icu.subject_id = me.subject_id 
    AND icu.hadm_id = me.hadm_id
WHERE icu.hadm_id IN {df_resp_inf_icu_hadm_ids}
     AND icu.hadm_id IS NOT NULL 
;



"""


resp_micro_hosp_icu_inf_df = pd.read_sql_query(query, con=engine)

In [32]:
hh.dxx(resp_micro_hosp_icu_inf_df)

11.3k Unique Patient IDs (11348)
12.9k Unique Admission IDs (12922)
15.7k Unique ICU Stay IDs (15723)
421.7k Rows, shape: (421739, 18)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,microevent_id,chartdate,charttime,spec_type_desc,test_name,org_name,ab_name,interpretation,dilution_value,comments
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,datetime64[ns],datetime64[ns],object,object,object,object,object,float64,object
NotNA | NA,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,156805 | 264934,127899 | 293840,127899 | 293840,121517 | 300222,356177 | 65562
nunique,11348,12922,15723,14,14,15723,15723,15561,289021,28540,121843,72,128,312,47,6,26,264
0,10011938,23798746,31780787,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2133-08-13 09:48:53,2133-09-01 18:22:42,19.356817,5128,2133-09-17 00:00:00,2133-09-17 13:05:00,BLOOD CULTURE,"Blood Culture, Routine",None,None,None,nan,NO GROWTH.
1,10011938,23798746,31780787,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2133-08-13 09:48:53,2133-09-01 18:22:42,19.356817,5139,2133-10-01 00:00:00,2133-10-01 23:17:00,BLOOD CULTURE,"Blood Culture, Routine",None,None,None,nan,NO GROWTH.
2,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2602,2135-01-04 00:00:00,2135-01-04 13:41:00,PLEURAL FLUID,GRAM STAIN,None,None,None,nan,NO POLYMORPHONUCLEAR LEUKOCYTES SEEN. NO MICROORGANISMS SEEN.


In [34]:
# hh.parq(resp_micro_hosp_icu_inf_df,'resp_micro_hosp_icu_inf_df_')

In [36]:
resp_micro_hosp_icu_inf_df=hh.load_data('./parq/resp_micro_hosp_icu_inf_df_23Jan26_2050.parquet')
hh.dxx(resp_micro_hosp_icu_inf_df)

11.3k Unique Patient IDs (11348)
12.9k Unique Admission IDs (12922)
15.7k Unique ICU Stay IDs (15723)
421.7k Rows, shape: (421739, 18)



,subject_id,hadm_id,stay_id,first_careunit,last_careunit,intime,outtime,los,microevent_id,chartdate,charttime,spec_type_desc,test_name,org_name,ab_name,interpretation,dilution_value,comments
dtype,int64,int64,int64,object,object,datetime64[ns],datetime64[ns],float64,int64,datetime64[ns],datetime64[ns],object,object,object,object,object,float64,object
NotNA | NA,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,421739 | 0,156805 | 264934,127899 | 293840,127899 | 293840,121517 | 300222,356177 | 65562
nunique,11348,12922,15723,14,14,15723,15723,15561,289021,28540,121843,72,128,312,47,6,26,264
0,10011938,23798746,31780787,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2133-08-13 09:48:53,2133-09-01 18:22:42,19.356817,5128,2133-09-17 00:00:00,2133-09-17 13:05:00,BLOOD CULTURE,"Blood Culture, Routine",None,None,None,nan,NO GROWTH.
1,10011938,23798746,31780787,Cardiac Vascular Intensive Care Unit (CVICU),Cardiac Vascular Intensive Care Unit (CVICU),2133-08-13 09:48:53,2133-09-01 18:22:42,19.356817,5139,2133-10-01 00:00:00,2133-10-01 23:17:00,BLOOD CULTURE,"Blood Culture, Routine",None,None,None,nan,NO GROWTH.
2,10005817,28661809,31316840,Medical/Surgical Intensive Care Unit (MICU/SICU),Medical/Surgical Intensive Care Unit (MICU/SICU),2135-01-03 21:55:32,2135-01-19 21:16:23,15.972812,2602,2135-01-04 00:00:00,2135-01-04 13:41:00,PLEURAL FLUID,GRAM STAIN,None,None,None,nan,NO POLYMORPHONUCLEAR LEUKOCYTES SEEN. NO MICROORGANISMS SEEN.


In [ ]:
query = f"""
-- Get respiratory infection ICU patients with any microbiology record during ICU stay (raw data)
SELECT 
    icu.subject_id,
    icu.hadm_id,
    icu.stay_id,
    icu.first_careunit,
    icu.last_careunit,
    icu.intime,
    icu.outtime,
    icu.los,
    me.microevent_id,
    me.chartdate,
    me.charttime,
    me.spec_type_desc,
    me.test_name,
    me.org_name,
    me.ab_name,
    me.interpretation,
    me.dilution_value,
    me.comments
FROM mimiciv_icu.icustays icu
INNER JOIN mimiciv_hosp.microbiologyevents me 
    ON icu.subject_id = me.subject_id 
    AND icu.hadm_id = me.hadm_id
   AND me.charttime BETWEEN icu.intime AND icu.outtime
WHERE icu.hadm_id IN {df_resp_inf_icu_hadm_ids}
     AND icu.hadm_id IS NOT NULL 
;



"""


resp_micro_icu_inf_df = pd.read_sql_query(query, con=engine)

In [90]:
hh.dx(resp_micro_icu_inf_df)

10.7k Unique Patient IDs (10722)
12.2k Unique Admission IDs (12150)
13.9k Unique ICU Stay IDs (13891)
205.8k Rows, shape: (205772, 18)



In [106]:
df_resp_inf_micro_icu_adm_ids = tuple(int(x) for x in resp_micro_icu_inf_df.hadm_id.unique())


In [ ]:
# resp_micro_icu_inf_df.to_parquet("resp_micro_icu_inf_df.parquet", engine="fastparquet")
resp_micro_icu_inf_df = pd.read_parquet("resp_micro_icu_inf_df.parquet", engine="fastparquet")

### ast icu

In [104]:
query = f"""
--- Get microbiology/AST results specifically for ICU respiratory patients
SELECT 
    me.subject_id,
    me.hadm_id,
    icu.stay_id,  -- ICU stay identifier
    icu.first_careunit,
    me.charttime,
    me.spec_type_desc,
    me.test_name,
    me.org_name,  -- Organism found
    me.ab_name,   -- Antibiotic tested
    me.dilution_value,
    me.interpretation,  -- S/I/R (Sensitive/Intermediate/Resistant)
    me.comments
FROM mimiciv_hosp.microbiologyevents me
INNER JOIN mimiciv_icu.icustays icu 
    ON me.subject_id = icu.subject_id 
    AND me.hadm_id = icu.hadm_id
    AND me.charttime BETWEEN icu.intime AND icu.outtime  -- Culture taken during ICU stay
WHERE me.charttime BETWEEN icu.intime AND icu.outtime  -- Culture taken during ICU stay
AND me.hadm_id IN {df_resp_inf_icu_hadm_ids}
     AND me.hadm_id IS NOT NULL 
     AND me.ab_name IS NOT NULL
    AND me.interpretation IS NOT NULL
;


"""


resp_ast_icu_inf_df = pd.read_sql_query(query, con=engine)

In [98]:
resp_ast_icu_inf_df.to_parquet("resp_ast_icu_inf_df.parquet", engine="fastparquet")


In [99]:
df_resp_inf_ast_icu_ids = tuple(int(x) for x in resp_ast_icu_inf_df.subject_id.unique())

In [100]:
df_resp_inf_ast_icu_adm_ids = tuple(int(x) for x in resp_ast_icu_inf_df.hadm_id.unique())


In [105]:
hh.dx(resp_ast_icu_inf_df)

3.2k Unique Patient IDs (3241)
3.5k Unique Admission IDs (3501)
3.7k Unique ICU Stay IDs (3698)
63.5k Rows, shape: (63461, 12)



In [ ]:
query = f"""
--- Get microbiology/AST results specifically for ICU respiratory patients
SELECT 
    me.subject_id,
    me.hadm_id,
    icu.stay_id,  -- ICU stay identifier
    icu.first_careunit,
    me.charttime,
    me.spec_type_desc,
    me.test_name,
    me.org_name,  -- Organism found
    me.ab_name,   -- Antibiotic tested
    me.dilution_value,
    me.interpretation,  -- S/I/R (Sensitive/Intermediate/Resistant)
    me.comments
FROM mimiciv_hosp.microbiologyevents me
INNER JOIN mimiciv_icu.icustays icu 
    ON me.subject_id = icu.subject_id 
    AND me.hadm_id = icu.hadm_id
    AND me.charttime BETWEEN icu.intime AND icu.outtime  -- Culture taken during ICU stay
WHERE me.charttime BETWEEN icu.intime AND icu.outtime  -- Culture taken during ICU stay
AND me.subject_id IN {df_resp_inf_icu_hadm_ids}
     AND me.hadm_id IS NOT NULL 
     AND me.ab_name IS NOT NULL
    AND me.interpretation IS NOT NULL
;


"""


resp_ast_icu_inf_df = pd.read_sql_query(query, con=engine)

In [ ]:
# resp_ast_icu_df.to_parquet("resp_ast_icu_df.parquet")
resp_ast_icu_df = pd.read_parquet("resp_ast_icu_df.parquet")

### ast results

In [ ]:
hh.dx(resp_ast_icu_inf_results_df)


In [ ]:
# resp_ast_icu_inf_results_df.to_parquet("resp_ast_icu_inf_results_df.parquet")
resp_ast_icu_inf_results_df = pd.read_parquet("resp_ast_icu_inf_results_df.parquet")

### prescriptions_ icu

In [107]:
query = f"""
-- Hospital Module: Prescriptions filtered to ICU time windows
SELECT 
    pr.subject_id,
    pr.hadm_id,
    icu.stay_id,
    icu.first_careunit,
    icu.last_careunit,
    icu.intime  AS icu_intime,
    icu.outtime AS icu_outtime,
    pr.pharmacy_id,
    pr.starttime,
    pr.stoptime,
    pr.drug_type,
    pr.drug,
    pr.formulary_drug_cd,
    pr.gsn,
    pr.ndc,
    pr.prod_strength,
    pr.form_rx,
    pr.dose_val_rx,
    pr.dose_unit_rx,
    pr.form_val_disp,
    pr.form_unit_disp,
    pr.doses_per_24_hrs,
    pr.route
FROM mimiciv_hosp.prescriptions pr
JOIN mimiciv_icu.icustays icu
  ON icu.subject_id = pr.subject_id
 AND icu.hadm_id    = pr.hadm_id
 -- time-overlap between prescription and ICU stay (handles NULL stoptime/outtime)
 AND pr.starttime < COALESCE(icu.outtime, pr.starttime + interval '1 second')
 AND COALESCE(pr.stoptime, pr.starttime) > icu.intime
WHERE pr.hadm_id IS NOT NULL
  AND pr.hadm_id IN {df_resp_inf_icu_hadm_ids}
;
"""

resp_pres_icu_inf_results_df = pd.read_sql_query(query, con=engine)

In [110]:
resp_pres_icu_inf_results_df.to_parquet("resp_pres_icu_inf_results_df.parquet", engine="fastparquet")
# resp_pres_icu_inf_results_df=pd.read_parquet("resp_pres_icu_inf_results_df.parquet")

In [ ]:
hh.dx(resp_pres_icu_inf_results_df)

#### prescriptions in ast

In [ ]:
query = f"""
-- Hospital Module: Prescriptions filtered to ICU time windows
SELECT 
    pr.subject_id,
    pr.hadm_id,
    icu.stay_id,
    icu.first_careunit,
    icu.last_careunit,
    icu.intime  AS icu_intime,
    icu.outtime AS icu_outtime,
    pr.pharmacy_id,
    pr.starttime,
    pr.stoptime,
    pr.drug_type,
    pr.drug,
    pr.formulary_drug_cd,
    pr.gsn,
    pr.ndc,
    pr.prod_strength,
    pr.form_rx,
    pr.dose_val_rx,
    pr.dose_unit_rx,
    pr.form_val_disp,C
    pr.form_unit_disp,
    pr.doses_per_24_hrs,
    pr.route
FROM mimiciv_hosp.prescriptions pr
JOIN mimiciv_icu.icustays icu
  ON icu.subject_id = pr.subject_id
 AND icu.hadm_id    = pr.hadm_id
 -- time-overlap between prescription and ICU stay (handles NULL stoptime/outtime)
 AND pr.starttime < COALESCE(icu.outtime, pr.starttime + interval '1 second')
 AND COALESCE(pr.stoptime, pr.starttime) > icu.intime
WHERE pr.hadm_id IS NOT NULL
  AND pr.subject_id IN {df_resp_inf_ast_icu_ids}
;
"""

resp_ast_pres_icu_inf_results_df = pd.read_sql_query(query, con=engine)

In [ ]:
hh.dxx(resp_ast_pres_icu_inf_results_df)

### emar_icu

In [108]:
query = f"""
-- eMAR + eMAR_DETAIL for dose-level fields (ICU-only, time-aligned to the ICU stay)
SELECT
  em.subject_id,
  em.hadm_id,
  icu.stay_id,
  icu.first_careunit,
  icu.last_careunit,
  icu.intime  AS icu_intime,
  icu.outtime AS icu_outtime,
  em.emar_id,
  em.emar_seq,
  em.charttime,
  em.medication,
  ed.parent_field_ordinal,
  ed.administration_type,
  ed.dose_due,
  ed.dose_due_unit,
  ed.dose_given,
  ed.dose_given_unit,
  ed.product_code,
  ed.product_description
FROM mimiciv_hosp.emar em
JOIN mimiciv_hosp.emar_detail ed
  ON em.subject_id = ed.subject_id
 AND em.emar_id    = ed.emar_id
 AND em.emar_seq   = ed.emar_seq
-- Restrict to ICU patients and align eMAR administrations to ICU stay times
JOIN mimiciv_icu.icustays icu
  ON icu.subject_id = em.subject_id
 AND icu.hadm_id    = em.hadm_id
 AND em.charttime  >= icu.intime
 AND (icu.outtime IS NULL OR em.charttime <= icu.outtime)
WHERE em.hadm_id IS NOT NULL
  AND em.hadm_id IN {df_resp_inf_icu_hadm_ids}
;
"""

resp_emar_icu_inf_results_df = pd.read_sql_query(query, con=engine)

In [111]:
hh.dx(resp_emar_icu_inf_results_df)

6.7k Unique Patient IDs (6735)
7.4k Unique Admission IDs (7447)
9.1k Unique ICU Stay IDs (9143)
5.0M Rows, shape: (4980994, 19)



In [112]:
resp_emar_icu_inf_results_df.to_parquet("resp_emar_icu_inf_results_df.parquet", engine="fastparquet")

In [ ]:
resp_emar_icu_inf_results_df=pd.read_parquet("resp_emar_icu_inf_results_df.parquet")

#### emar in ast df

In [ ]:
query = f"""
-- eMAR + eMAR_DETAIL for dose-level fields (ICU-only, time-aligned to the ICU stay)
SELECT
  em.subject_id,
  em.hadm_id,
  icu.stay_id,
  icu.first_careunit,
  icu.last_careunit,
  icu.intime  AS icu_intime,
  icu.outtime AS icu_outtime,
  em.emar_id,
  em.emar_seq,
  em.charttime,
  em.medication,
  ed.parent_field_ordinal,
  ed.administration_type,
  ed.dose_due,
  ed.dose_due_unit,
  ed.dose_given,
  ed.dose_given_unit,
  ed.product_code,
  ed.product_description
FROM mimiciv_hosp.emar em
JOIN mimiciv_hosp.emar_detail ed
  ON em.subject_id = ed.subject_id
 AND em.emar_id    = ed.emar_id
 AND em.emar_seq   = ed.emar_seq
-- Restrict to ICU patients and align eMAR administrations to ICU stay times
JOIN mimiciv_icu.icustays icu
  ON icu.subject_id = em.subject_id
 AND icu.hadm_id    = em.hadm_id
 AND em.charttime  >= icu.intime
 AND (icu.outtime IS NULL OR em.charttime <= icu.outtime)
WHERE em.hadm_id IS NOT NULL
  AND em.subject_id IN {df_resp_inf_ast_icu_ids}
;
"""

resp_ast_emar_icu_inf_results_df = pd.read_sql_query(query, con=engine)

In [ ]:
hh.dx(resp_ast_emar_icu_inf_results_df)

### INPUT EVENTS

In [7]:
df_resp_inf_icu_final = pd.read_parquet("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/parq/df_resp_inf_icu_final.parq", engine="fastparquet")
df_resp_inf_icu_hadm_ids = tuple(df_resp_inf_icu_final.hadm_id.unique().tolist())
df_resp_inf_icu_subject_ids = tuple(df_resp_inf_icu_final.subject_id.unique().tolist())

In [10]:
len(df_resp_inf_icu_subject_ids)

11905

In [8]:
query = f"""
-- ICU inputevents antibiotic infusions for respiratory-infection admissions (raw, original columns)
SELECT
  ie.subject_id,
  ie.hadm_id,
  ie.stay_id,
  ie.starttime,
  ie.endtime,
  ie.storetime,
  ie.itemid,
  di.label,
  ie.amount,
  ie.amountuom,
  ie.rate,
  ie.rateuom,
  ie.orderid,
  ie.linkorderid,
  ie.ordercategoryname,
  ie.secondaryordercategoryname,
  ie.ordercomponenttypedescription,
  ie.ordercategorydescription,
  ie.patientweight,
  ie.totalamount,
  ie.totalamountuom,
  ie.isopenbag,
  ie.continueinnextdept,
  ie.statusdescription,
  ie.originalamount,
  ie.originalrate
FROM mimiciv_icu.inputevents ie
LEFT JOIN mimiciv_icu.d_items di ON ie.itemid = di.itemid
WHERE ie.hadm_id IS NOT NULL
  AND ie.subject_id IN {df_resp_inf_icu_subject_ids};




"""


all_inputevents_icu_inf_results_df = pd.read_sql_query(query, con=engine)

In [9]:
hh.dx(all_inputevents_icu_inf_results_df)

10.8k Unique Patient IDs (10786)
19.0k Unique Admission IDs (18959)
22.0k Unique ICU Stay IDs (22027)
4.4M Rows, shape: (4393455, 26)



In [11]:
hh.parq(all_inputevents_icu_inf_results_df,'all_inputevents_icu_inf_results_df_')

File saved at: all_inputevents_icu_inf_results_df_02Mar26_0400.parquet


In [13]:
all_inputevents_icu_inf_results_df= hh.load_data('./parq/all_inputevents_icu_inf_results_df_02Mar26_0400.parquet')

In [115]:
resp_inputevents_icu_inf_results_df.to_parquet("resp_inputevents_icu_inf_results_df.parquet", engine="fastparquet")


In [ ]:
resp_inputevents_icu_inf_results_df= pd.read_parquet("resp_inputevents_icu_inf_results_df.parquet")

#### inputevents in ast

In [ ]:
query = f"""
-- ICU inputevents antibiotic infusions for respiratory-infection admissions (raw, original columns)
SELECT
  ie.subject_id,
  ie.hadm_id,
  ie.stay_id,
  ie.starttime,
  ie.endtime,
  ie.storetime,
  ie.itemid,
  di.label,
  ie.amount,
  ie.amountuom,
  ie.rate,
  ie.rateuom,
  ie.orderid,
  ie.linkorderid,
  ie.ordercategoryname,
  ie.secondaryordercategoryname,
  ie.ordercomponenttypedescription,
  ie.ordercategorydescription,
  ie.patientweight,
  ie.totalamount,
  ie.totalamountuom,
  ie.isopenbag,
  ie.continueinnextdept,
  ie.statusdescription,
  ie.originalamount,
  ie.originalrate
FROM mimiciv_icu.inputevents ie
LEFT JOIN mimiciv_icu.d_items di ON ie.itemid = di.itemid
WHERE ie.hadm_id IS NOT NULL
  AND ie.subject_id IN {df_resp_inf_ast_icu_ids};




"""


resp_ast_inputevents_icu_inf_results_df = pd.read_sql_query(query, con=engine)

In [ ]:
hh.dx(resp_ast_inputevents_icu_inf_results_df)
      

# Cohort funnel: from overall to final modeling cohort
This section summarizes the stepwise cohort attrition using MIMIC-IV tables and the materialized modality datasets. We report counts of unique subjects, admissions (HADM_ID), and ICU stays (STAY_ID) for each stage, and the final intersections (5-set and AST-inclusive).

In [17]:
import pandas as pd
from sqlalchemy import text

# Helper to format counts safely
def _counts(df, subj_col="subject_id", hadm_col="hadm_id", stay_col="stay_id"):
    d = {}
    if subj_col in df.columns:
        d["n_subjects"] = int(df[subj_col].nunique())
    else:
        d["n_subjects"] = None
    if hadm_col in df.columns:
        d["n_admissions"] = int(df[hadm_col].nunique())
    else:
        d["n_admissions"] = None
    if stay_col in df.columns:
        d["n_stays"] = int(df[stay_col].nunique())
    else:
        d["n_stays"] = None
    return d

# 1) Overall population and ICU totals (direct SQL counts)
sql_overall = text("""
SELECT
    (SELECT COUNT(DISTINCT subject_id) FROM mimiciv_hosp.admissions) AS hosp_subjects,
    (SELECT COUNT(DISTINCT hadm_id)    FROM mimiciv_hosp.admissions) AS hosp_admissions,
    (SELECT COUNT(DISTINCT subject_id) FROM mimiciv_icu.icustays)     AS icu_subjects,
    (SELECT COUNT(DISTINCT hadm_id)    FROM mimiciv_icu.icustays)     AS icu_admissions,
    (SELECT COUNT(DISTINCT stay_id)    FROM mimiciv_icu.icustays)     AS icu_stays
""")
overall_row = pd.read_sql(sql_overall, con=engine).iloc[0].to_dict()

# 2) Respiratory infections (hospital diagnoses) — admissions-level
sql_resp_hosp = text("""
SELECT
    COUNT(DISTINCT subject_id) AS resp_subjects,
    COUNT(DISTINCT hadm_id)    AS resp_admissions
FROM mimiciv_hosp.diagnoses_icd di
WHERE
  (di.icd_version = 10 AND UPPER(di.icd_code) ~ '^(J0[0-6]|J09|J1[0-8]|J2[0-2]|J8[56])')
  OR
  (di.icd_version = 9  AND di.icd_code ~ '^(46[0-6]|48[0-8])')
""")
resp_hosp_row = pd.read_sql(sql_resp_hosp, con=engine).iloc[0].to_dict()

# 3) Respiratory infections with ICU stay (join diagnoses to ICU)
sql_resp_icu = text("""
SELECT
  COUNT(DISTINCT i.subject_id) AS resp_icu_subjects,
  COUNT(DISTINCT i.hadm_id)    AS resp_icu_admissions,
  COUNT(DISTINCT i.stay_id)    AS resp_icu_stays
FROM mimiciv_icu.icustays i
JOIN mimiciv_hosp.diagnoses_icd di
  ON di.subject_id = i.subject_id AND di.hadm_id = i.hadm_id
WHERE
  (di.icd_version = 10 AND UPPER(di.icd_code) ~ '^(J0[0-6]|J09|J1[0-8]|J2[0-2]|J8[56])')
  OR
  (di.icd_version = 9  AND di.icd_code ~ '^(46[0-6]|48[0-8])')
""")
resp_icu_row = pd.read_sql(sql_resp_icu, con=engine).iloc[0].to_dict()

# 4) Load saved ICU respiratory cohort (if present) and compute counts
try:
    df_resp_icu = pd.read_parquet("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/df_resp_inf_icu_final.parq", engine="fastparquet")
    resp_icu_saved = _counts(df_resp_icu)
except Exception:
    df_resp_icu = None
    resp_icu_saved = {"n_subjects": None, "n_admissions": None, "n_stays": None}

# 5) Modality availability within ICU (materialized datasets)
modality_files = {
    "Labs (ICU)": "/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/resp_lab_test_icu_inf_df.parq",
    "Microbiology (ICU)": "/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/resp_micro_icu_inf_df.parquet",
    "AST (ICU subset)": "/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/resp_ast_icu_inf_df.parquet",
    "Prescriptions (ICU)": "/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/resp_pres_icu_inf_results_df.parquet",
    "EMAR (ICU)": "/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/resp_emar_icu_inf_results_df.parquet",
    "InputEvents (ICU)": "/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/resp_inputevents_icu_inf_results_df.parquet",
}
mod_counts = {}
for label, path in modality_files.items():
    try:
        dfm = pd.read_parquet(path, engine="fastparquet") if path.endswith((".parq", ".parquet")) else pd.read_parquet(path)
        mod_counts[label] = _counts(dfm)
        mod_counts[label]["_hadm_set"] = set(dfm["hadm_id"].dropna().astype(int).unique()) if "hadm_id" in dfm.columns else set()
    except Exception:
        mod_counts[label] = {"n_subjects": None, "n_admissions": None, "n_stays": None, "_hadm_set": set()}

# 6) Intersections (5-set and AST-inclusive)
try:
    hadm_5k = set(pd.read_csv("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/common_hadm_ids_5k.csv")["hadm_id"].astype(int).unique())
except Exception:
    hadm_5k = set()
try:
    hadm_1k = set(pd.read_csv("/Users/gnaanikko.pa/Documents/Academic /MIMIC/model_building/outputs/common_hadm_ids_1k.csv")["hadm_id"].astype(int).unique())
except Exception:
    hadm_1k = set()

# Map hadm -> subject for subject counts at intersection levels
try:
    df_adm = pd.read_parquet("df_admissions.parq", engine="fastparquet")
    hadm_to_subj = df_adm.set_index("hadm_id")["subject_id"].to_dict()
    subj_5k = len({hadm_to_subj[h] for h in hadm_5k if h in hadm_to_subj})
    subj_1k = len({hadm_to_subj[h] for h in hadm_1k if h in hadm_to_subj})
except Exception:
    subj_5k = None
    subj_1k = None

# Build funnel table
rows = []
rows.append({
    "Step": "All ICU (icustays)",
    "n_subjects": int(overall_row.get("icu_subjects")),
    "n_admissions": int(overall_row.get("icu_admissions")),
    "n_stays": int(overall_row.get("icu_stays")),
})
rows.append({
    "Step": "Respiratory diagnoses (hospital, any care)",
    "n_subjects": int(resp_hosp_row.get("resp_subjects")),
    "n_admissions": int(resp_hosp_row.get("resp_admissions")),
    "n_stays": None,
})
rows.append({
    "Step": "Respiratory diagnoses with ICU stay (join)",
    "n_subjects": int(resp_icu_row.get("resp_icu_subjects")),
    "n_admissions": int(resp_icu_row.get("resp_icu_admissions")),
    "n_stays": int(resp_icu_row.get("resp_icu_stays")),
})
rows.append({
    "Step": "Saved ICU respiratory cohort (df_resp_inf_icu_final)",
    "n_subjects": resp_icu_saved["n_subjects"],
    "n_admissions": resp_icu_saved["n_admissions"],
    "n_stays": resp_icu_saved["n_stays"],
})
for label in ["Labs (ICU)", "Microbiology (ICU)", "AST (ICU subset)", "Prescriptions (ICU)", "EMAR (ICU)", "InputEvents (ICU)"]:
    rows.append({
        "Step": f"Has {label}",
        "n_subjects": mod_counts[label]["n_subjects"],
        "n_admissions": mod_counts[label]["n_admissions"],
        "n_stays": mod_counts[label]["n_stays"],
    })
rows.append({
    "Step": "5-set intersection (Micro ∩ Pres ∩ EMAR ∩ Input ∩ Labs)",
    "n_subjects": subj_5k,
    "n_admissions": len(hadm_5k) if hadm_5k else None,
    "n_stays": None,
})
rows.append({
    "Step": "AST-inclusive intersection (AST ∩ Pres ∩ EMAR ∩ Input ∩ Labs)",
    "n_subjects": subj_1k,
    "n_admissions": len(hadm_1k) if hadm_1k else None,
    "n_stays": None,
})

funnel_df = pd.DataFrame(rows)
from IPython.display import display
print("Cohort funnel (counts of unique entities):")
display(funnel_df)
print("\nMarkdown table:\n")
print(funnel_df.to_markdown(index=False))

# Save CSV
# try:
#     funnel_df.to_csv("model_building/outputs/cohort_funnel_counts.csv", index=False)
# except Exception:
#     pass

Cohort funnel (counts of unique entities):


,Step,n_subjects,n_admissions,n_stays
0,All ICU (icustays),65366.0,85242,94458.0
1,"Respiratory diagnoses (hospital, any care)",27480.0,37627,NaN
2,Respiratory diagnoses with ICU stay (join),11905.0,13611,16438.0
3,Saved ICU respiratory cohort (df_resp_inf_icu_...,11905.0,13611,16438.0
4,Has Labs (ICU),11864.0,13549,NaN
5,Has Microbiology (ICU),10722.0,12150,13891.0
6,Has AST (ICU subset),3241.0,3501,3698.0
7,Has Prescriptions (ICU),11869.0,13568,16378.0
8,Has EMAR (ICU),6735.0,7447,9143.0
9,Has InputEvents (ICU),10617.0,12178,14523.0



Markdown table:

| Step                                                          |   n_subjects |   n_admissions |   n_stays |
|:--------------------------------------------------------------|-------------:|---------------:|----------:|
| All ICU (icustays)                                            |        65366 |          85242 |     94458 |
| Respiratory diagnoses (hospital, any care)                    |        27480 |          37627 |       nan |
| Respiratory diagnoses with ICU stay (join)                    |        11905 |          13611 |     16438 |
| Saved ICU respiratory cohort (df_resp_inf_icu_final)          |        11905 |          13611 |     16438 |
| Has Labs (ICU)                                                |        11864 |          13549 |       nan |
| Has Microbiology (ICU)                                        |        10722 |          12150 |     13891 |
| Has AST (ICU subset)                                          |         3241 |           3501 |     